In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import scipy.linalg as la
from scipy.integrate import solve_ivp
from scipy.optimize import root_scalar

In [58]:
# -- Definition of constants --
b = 4.0
V0 = 50.0
Nmax = 50
Emax = 50.0

In [ ]:
# -- Definition of functions for Matrix Method --

def matrix_method(V, Nmax, b, dim='1D', m_quant=0):
    """_summary_

    Args:
        V (_type_): _description_
        Nmax (_type_): _description_
        b (_type_): _description_
        dim (str, optional): _description_. Defaults to '1D'.
        m_quant (int, optional): _description_. Defaults to 0.

    Returns:
        _type_: _description_
    """
    def basis_func(n, x):
        return np.sqrt(2.0/b) * np.sin(n * np.pi * x / b)
    
    H = np.zeros((Nmax, Nmax))
    
    # First the elements in the diagonal
    E_0 = [(n**2 * np.pi**2) / (2.0 * b**2) for n in range(1, Nmax + 1)]
    
    # Populate the diagonal elements in the matrix
    for i in range(Nmax):
        H[i,i] = E_0[i]
        
    def Veff(x):
        if dim =='1D':
            return V(x)
        elif dim =='2D_radial':
            if x <= 1e-12:
                return V(x)
            centrifugal_term = (m_quant**2 - 0.25) / (2.0 * x**2)
            return V(x) + centrifugal_term
    
    # Now we will compute the other matrix elements by using nested loops and
    # numerical integration. 
    for n in range(Nmax):
        n_index = n + 1
        for m in range(Nmax): 
            m_index = m + 1
            integrand = lambda x: basis_func(n_index, x) * Veff(x) * basis_func(m_index, x)
            V_nm, _ = quad(integrand, 0.0, b, limit=100)
            H[n, m] += V_nm
            
    energies, vectors = la.eigh(H)
    return H, energies, vectors

def reconstruct_wavefunction(vectors, energies, b, Nmax, Emax):
    """
        _summary_
    Args:
        vectors (_type_): _description_
        energies (_type_): _description_
        b (_type_): _description_
        Nmax (_type_): _description_
        Emax (_type_): _description_

    Returns:
        _type_: _description_
    """

    bound_state_indices = np.where(energies <= Emax)[0]
    bound_state_vectors = vectors[:, bound_state_indices]
    bound_state_energies = energies[bound_state_indices]
        
    x_grid = np.linspace(0, b, 1000)
    all_n_indices = np.arange(1, Nmax + 1)
    
    Phi = np.sqrt(2.0 / b) * np.sin(np.outer(x_grid, all_n_indices * np.pi / b))
    
    print(np.shape(Phi)) # This returns a 1000xNmax matrix with the basis functions evaluated at 1000 points in x.
    print(np.shape(bound_state_vectors)) # This returns a NmaxxNmax matrix withÇ
    
    Psi = Phi @ bound_state_vectors
    probability_density = np.abs(Psi)**2
    
    return x_grid, Psi, probability_density, bound_state_energies # This returns a 2D matrix with Nmax amount of functions for different energies eigenvalues for a 1000 values of x.


In [56]:
def plot_bound_states_grid(x_grid, Psi, prob_density, bound_energies, Emax, b=4.0, ncols=2):
    """
    Plots a grid of subplots for all bound states, displaying both 
    the wavefunction Psi(x) and probability density |Psi(x)|^2.
    """
    num_states = len(bound_energies)
    nrows = (num_states + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows), sharex=True)
    
    # Flatten axes for iteration (handles single-row edge case)
    axes = np.atleast_1d(axes).flatten()
    
    for k in range(num_states):
        ax = axes[k]
        
        # 1. Left Y-axis: Wavefunction Psi(x)
        color_psi = 'tab:blue'
        ax.set_ylabel('Psi(x)', color=color_psi, fontsize=11)
        ax.plot(x_grid, Psi[:, k], color=color_psi, linewidth=2, label='Psi(x)')
        ax.tick_params(axis='y', labelcolor=color_psi)
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.6)
        
        # 2. Right Y-axis: Probability Density |Psi(x)|^2
        ax2 = ax.twinx()
        color_prob = 'tab:red'
        ax2.set_ylabel('|Psi(x)|^2', color=color_prob, fontsize=11)
        ax2.plot(x_grid, prob_density[:, k], color=color_prob, linewidth=1.8, label='|Psi(x)|^2')
        ax2.fill_between(x_grid, 0, prob_density[:, k], color=color_prob, alpha=0.15)
        ax2.tick_params(axis='y', labelcolor=color_prob)
        
        # Subplot Title & Limits
        ax.set_title(f'Bound State {k+1}: E = {bound_energies[k]:.3f}', fontsize=12, fontweight='bold')
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.set_xlim(0, b)

    # Add x-axis label to the bottom row
    for ax in axes[-ncols:]:
        ax.set_xlabel('Position x', fontsize=11)

    # Remove unused empty subplots if num_states is odd
    for k in range(num_states, len(axes)):
        fig.delaxes(axes[k])

    plt.suptitle(f"Bound State Wavefunctions and Probability Densities (E <= {Emax})", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
def shooting_method(V, b, Emax=50.0, dim='1D', m_quant=0, n_scan=300, symmetric=True):
    """
    Generalized Shooting Method for finding bound state energies on [0, b].
    Mirrors the interface of matrix_method().

    Args:
        V (callable): Potential function V(x) defined on [0, b].
        b (float): Integration domain width [0, b] (centered at b/2 for symmetric 1D potentials).
        Emax (float, optional): Maximum energy for bound states (V0). Defaults to 50.0.
        dim (str, optional): '1D' or '2D_radial'. Defaults to '1D'.
        m_quant (int, optional): Angular momentum quantum number for '2D_radial'. Defaults to 0.
        n_scan (int, optional): Number of energy grid points to bracket roots. Defaults to 300.
        symmetric (bool, optional): If True (and dim=='1D'), exploits parity symmetry about x = b/2
                                    by integrating from b/2 to b. Defaults to True.

    Returns:
        bound_energies (np.ndarray): Array of bound state energy eigenvalues.
        state_parities (list): List of parity labels ('even', 'odd', or 'full') for reconstruction.
    """
    def Veff(x):
        if dim == '1D':
            return V(x)
        elif dim == '2D_radial':
            r = max(x, 1e-9)
            return V(r) + (m_quant**2 - 0.25) / (2.0 * r**2)

    def ode_rhs(x, y, E):
        # y[0] = Psi(x), y[1] = Psi'(x)
        return [y[1], -2.0 * (E - Veff(x)) * y[0]]

    E_grid = np.linspace(0.1, Emax - 1e-3, n_scan)
    found_states = []

    if dim == '1D' and symmetric:
        # Integrate from center (x = b/2) to right boundary (x = b)
        x_center = b / 2.0
        for parity in ['even', 'odd']:
            y0 = [1.0, 0.0] if parity == 'even' else [0.0, 1.0]
            
            def shoot_bc(E):
                sol = solve_ivp(ode_rhs, (x_center, b), y0, args=(E,), max_step=0.005, rtol=1e-8, atol=1e-8)
                return sol.y[0, -1]

            vals = [shoot_bc(E) for E in E_grid]
            for i in range(len(E_grid) - 1):
                if vals[i] * vals[i + 1] < 0:
                    res = root_scalar(shoot_bc, bracket=[E_grid[i], E_grid[i + 1]], method='brentq')
                    if res.converged:
                        found_states.append((res.root, parity))
    else:
        # Full-domain shooting from x_start to b (for asymmetric 1D or 2D_radial)
        x_start = 1e-6 if dim == '2D_radial' else 0.0
        
        def shoot_bc_full(E):
            if dim == '2D_radial':
                # Asymptotic behavior near r -> 0: u(r) ~ r^(|m| + 1/2)
                p = abs(m_quant) + 0.5
                y0 = [x_start**p, p * (x_start**(p - 1.0))]
            else:
                y0 = [0.0, 1.0]
            sol = solve_ivp(ode_rhs, (x_start, b), y0, args=(E,), max_step=0.005, rtol=1e-8, atol=1e-8)
            return sol.y[0, -1]

        vals = [shoot_bc_full(E) for E in E_grid]
        for i in range(len(E_grid) - 1):
            if vals[i] * vals[i + 1] < 0:
                res = root_scalar(shoot_bc_full, bracket=[E_grid[i], E_grid[i + 1]], method='brentq')
                if res.converged:
                    found_states.append((res.root, 'full'))

    # Sort states by ascending energy
    found_states.sort(key=lambda item: item[0])
    bound_energies = np.array([item[0] for item in found_states])
    state_parities = [item[1] for item in found_states]

    return bound_energies, state_parities

In [ ]:
def reconstruct_shooting_wavefunction(V, bound_energies, state_parities, b, dim='1D', m_quant=0, num_pts=1000):
    """
    Reconstructs and normalizes wavefunctions on [0, b] from shooting_method results.
    Matches the return signature of reconstruct_wavefunction().

    Returns:
        x_grid (np.ndarray): 1D array of x values on [0, b].
        Psi (np.ndarray): 2D array (num_pts x num_states) of normalized wavefunctions.
        probability_density (np.ndarray): 2D array of |Psi(x)|^2.
        bound_energies (np.ndarray): 1D array of bound state energies.
    """
    def Veff(x):
        if dim == '1D':
            return V(x)
        elif dim == '2D_radial':
            r = max(x, 1e-9)
            return V(r) + (m_quant**2 - 0.25) / (2.0 * r**2)

    def ode_rhs(x, y, E):
        return [y[1], -2.0 * (E - Veff(x)) * y[0]]

    psi_list = []

    if dim == '1D' and len(state_parities) > 0 and state_parities[0] in ['even', 'odd']:
        half_pts = num_pts // 2 + 1
        x_right = np.linspace(b / 2.0, b, half_pts)
        x_left = b - x_right[::-1][:-1]
        x_grid = np.concatenate([x_left, x_right])

        for E, parity in zip(bound_energies, state_parities):
            y0 = [1.0, 0.0] if parity == 'even' else [0.0, 1.0]
            sol = solve_ivp(ode_rhs, (b / 2.0, b), y0, args=(E,), t_eval=x_right, max_step=0.005, rtol=1e-8, atol=1e-8)
            psi_r = sol.y[0]
            psi_l = psi_r[::-1][:-1] if parity == 'even' else -psi_r[::-1][:-1]
            psi = np.concatenate([psi_l, psi_r])

            # Normalize wavefunction on [0, b]
            norm = np.sqrt(np.trapezoid(psi**2, x_grid))
            psi_list.append(psi / norm)
    else:
        x_start = 1e-6 if dim == '2D_radial' else 0.0
        x_grid = np.linspace(x_start, b, num_pts)

        for E in bound_energies:
            if dim == '2D_radial':
                p = abs(m_quant) + 0.5
                y0 = [x_start**p, p * (x_start**(p - 1.0))]
            else:
                y0 = [0.0, 1.0]
            sol = solve_ivp(ode_rhs, (x_start, b), y0, args=(E,), t_eval=x_grid, max_step=0.005, rtol=1e-8, atol=1e-8)
            psi = sol.y[0]

            # Normalize wavefunction on [0, b]
            norm = np.sqrt(np.trapezoid(psi**2, x_grid))
            psi_list.append(psi / norm)

    Psi = np.column_stack(psi_list) if psi_list else np.empty((len(x_grid), 0))
    probability_density = np.abs(Psi)**2

    return x_grid, Psi, probability_density, bound_energies

In [ ]:
def single_well_potential(x):
    

In [ ]:
matrix_method(V, Nmax, b, dim='1D', m_quant=0)